# Practice Session PS05: Content-based recommendation

For this assignment we will build and use a content-based recommender system for movies. This content-based recommender will take as input a database of users and ratings on movies.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

⭐ **Regarding the usage of generative AI.** You may use generative AI to (a) understand how to do certain operations, and (b) generate short snippets of code, that you can incorporate into your practice after reviewing them. In that case, add a comment `#AI` before the snippet. You can also add `#AI (edited)` if you edited the code before incorporating it.

⚠️ It is forbidden to use generative AI in these practices for answering the text-based questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="#2680EB">Your name here</font>

E-mail: <font color="#2680EB">Your e-mail here</font>

AI used: <font color="#2680EB">Maker and model, or "none".</font>

Date: <font color="#2680EB">The current date here</font>

# 0. Load the movies dataset

This dataset contains over 9,500 movies from [The Movies Database](https://www.themoviedb.org/). It was originally [developed](https://github.com/metarank/msrd) for search tasks, but we will use it for content-based recommendation.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Required libraries
import pandas as pd   
import numpy as np  
import re
import random

# Plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt 

# Sci-kit specific libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

The following code, which you should leave as-is, loads the 3 dataset within three dataframes named `movies`,  `ratings` and `tags` considering the data structure explained above.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

FILENAME_MOVIES = "movies.csv.gz"

In [ ]:
# LEAVE AS-IS

# Load movies
movies = pd.read_csv(FILENAME_MOVIES, 
                    compression='gzip',
                    sep="\t", 
                    engine='python', 
                    encoding='utf-8')

# Display a random sample
print(f"Loaded {len(movies)} movies, here is a sample:")
display(movies.sample(3))

# 1. Create a function to find movies <font color="#2680EB">[10%]</font>

Create a function `find_movies` to easily find movies that contain a given keyword in its title. For instance `find_movies("Purple", movies)` should print:

```
id:  762975, title: Purple Hearts
id:     873, title: The Color Purple
```

*Tip:* to know if string *a* is contained in string *b*, you can ask `if a in b`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement find_movies** to find movies containing a keyword.</font>

In [ ]:
def find_movies(keyword, movies):
    # YOUR CODE HERE

find_movies("Purple", movies)

# 2. Distributions of genre and keywords

Tipically, a movie genre (e.g., "musical", "action", "romance") is one of the most important descriptor for a movie recommendation engine: a basic assumption is that films in the same genre should have similar contents.

The following code lists all genres in the dataset.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def print_genres(df):
    print(f"Movies         : {len(df)}" )
    print(f" Without genre : {df['genres'].isna().sum()}" )
    print(f" With genre    : {df['genres'].notna().sum()}" )
    print()

    
    all_genres = set()
    for s in df['genres'].dropna().str.split(',').values:
        all_genres = all_genres.union(set(s))
    print(all_genres)
    
print_genres(movies)
    

## 2.1. Prepare the "genre" column <font color="#2680EB">[15%]</font>

As you can see, the values in the "genre" column have various issues that might complicate drawing statistics and other conclusions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Indicate what problems do you see in the genre column, and how would you clean this column.**</font>

**Responses.**

What issues do you see in the list of genres below?

* YOUR ANSWER HERE

How would you clean this column?

* YOUR ANSWER HERE


<font size="+1" color="#2680EB">**Clean the genre column**. In particular, make sure "(no genres listed)" is NULL, and that there are no (near)duplicate genres.</font>

In [ ]:
movies1 = movies.copy()

# YOUR CODE HERE

print_genres(movies1)

The following plots frequency distribution of genres.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def plot_genres(df):
    
    # Convert genres into a list (i.e., "explode" it)
    exploded_genres = df['genres'].str.split(',').explode().str.strip()

    # Create frequency DataFrame
    genre_counts = exploded_genres.value_counts().reset_index()
    genre_counts.columns = ['genre', 'count']
    genre_counts['percent'] = (genre_counts['count'] / genre_counts['count'].sum()) * 100

    # Create barchart
    sns.barplot(data=genre_counts, x='percent', y='genre', legend=False)
    plt.title('Movies by genre')
    plt.xlabel('Percentage (%)')
    plt.ylabel('Genre')
    
    # Add vertical lines
    # The instructor used AI to set this grid (Gemini 3.6 Flash)
    max_percentage = int(np.ceil(genre_counts['percent'].max()))
    plt.xticks(np.arange(0, max_percentage + 1, 1 ))
    plt.grid(axis='x', alpha=0.7)
    
    plt.show()

plot_genres(movies1)

Let us merge some genres, with a focus on incorporating smaller genres into larger ones.

We will use the following to group some genres:

* "crime" (film-noir or crime)
* "musical" (music or musical)
* "sci-fi" (science fiction or sci-fi)
* "documentary" (history or documentary)
* "family" (children or family)
* "action" (war or action)

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Merge genres** using the specification above.</font>

In [ ]:
pairs_to_replace = [
    ('film-noir', 'crime'),
    ('music', 'musical'),
    ('science fiction', 'sci-fi'),
    ('history', 'documentary'),
    ('children', 'family'),
    ('war', 'action')
]

movies2 = movies1.copy()
# YOUR CODE HERE

plot_genres(movies2)

## 2.2. Merge genres and tags into keywords <font color="#2680EB">[15%]</font>

We will use **genres** and **tags** describing each movie. Collectively, we will call these **keywords**. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Create column "keywords".** It should contain a list, which should be the union of "genres" and "tags".</font>

In [ ]:
# Merge source1 and source2 (comma-separated) into target (list)
def merge_csv_arrays(df, source1, source2, target):
    
    # Copy the dataframe
    dfc = df.copy()
    
    # YOUR CODE HERE
    
    # Replaces missing values with an empty list
    dfc[target] = dfc[target].mask(dfc[target].isna(), [[] for _ in range(len(dfc))])
    return dfc

movies3 = merge_csv_arrays(movies2, "genres", "tags", "keywords")

Verify by running the next cell. It should print how genres and tags were transformed into keywords for a sample of movies.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Sample 3 random movies
for i in range(3):
    movie = movies3[['title', 'genres', 'tags', 'keywords']].sample().iloc[0]

    # Print genres, tags → keywords
    print(f"Title: {movie['title']}")
    print(f" genres={movie['genres']}, tags={movie['tags']} => keywords={movie['keywords']}")
    print()


# 3. Content-based recommendation

A content-based recommender works with data that describes the movie (or content). In our case, the **keywords**.

We *could* extend the content-based taking into account the *overview* or any other text-based characteristics of the movie. We could also extend it using other attributes of the movie, even if they are not text. Content-based recommendations just means that we somehow compute the similarity between the items and then recommend the most similar items.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 3.1. Compute TF-IDF vectors

**Term Frequency (TF)** and **Inverse Document Frequency (IDF)** are techniques used in information retrieval to determine the **importance** of a word. In our use case, content-based recommendation, TF-IDF will be used to measure the importance of each genre in each movie. 

TF is simply the frequency of a word in a document, IDF is the inverse of the document frequency among the whole corpus of documents. Usually, TF and IDF are combined to measure the relevance of a word or term across different documents.

In a nutshell, TF-IDF weights infrequent words (in our case genres) more heavily. This means that two movies that share many uncommon genres will be considered more similar than two movies that share many common genres.

The TF-IDF score is computed as:

**tfidf(keyword, document) = TF(keyword, document) x IDF(keyword)**

Where:

* **TF(keyword, doc)** is the number of times the *keyword* appears in the *document*
* **IDF(keyword)** is **log(N/docfreq(keyword))** where **N** is the total number of documents and **docfreq(keyword)** is the number of documents containing the **keyword**

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Next, we use [sklearn.feature_extraction.text.TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to compute the tf-idf vectors on this column. These are stored in `tfidf_matrix`.

The resulting matrix should have 9751 rows (one per movie) and 7875 columns (one per unique keywod). We verify this using `tfidf_matrix.shape`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

tfidf = TfidfVectorizer(analyzer=lambda x: x, min_df=1)
tfidf_matrix = tfidf.fit_transform(movies3['keywords'])
print(f"The TF-IDF matrix has {tfidf_matrix.shape[0]} rows (movies)")
print(f"The TF-IDF matrix has {tfidf_matrix.shape[1]} columns (distinct keywords)")

## 3.2. Calculate most similar movies <font color="#2680EB">[20%]</font>

As previously explained, the calculated TF-IDF from **keywords** will be the variable to describe the content; i.e. in our case, the movie. 

The **similarity** will be measured considering the angle between the vectors generated after the TF-IDF process. Each movie becomes a vector of attributes in an **n-dimensional space** where the angles between the vectors **determines the similarity**. 

The method of calculating this relative measure is taking the cosine of the angle between two vectors. The vectors are length normalized and then the cosine calculation is simply the sum-product of vectors. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

The next code, which you should leave as-is, uses sklearn's [linear_kernel](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.linear_kernel.html) function, which is a fast way of computing these similarities on a matrix.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"The similarity matrix has {cosine_sim.shape[0]} rows and {cosine_sim.shape[1]} columns")
print(cosine_sim)

It is easy to confuse movie ids (column "id") with the number of a row or column (0 to N-1), which is in the index of the `movie` dataframe. Let us address that.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Create functions "get_title", "get_index_by_movieid",  "get_movieid_by_index".** They should, respectively, return the title, the index of the row of a movie with id=movieid, and the movieid of the row of a movie indexed by index. In all cases, make sure they return a null ("None") if this is not found.</font>

In [ ]:
def get_title_by_movieid(df, movie_id):
    # YOUR CODE HERE
    return return_value

def get_index_by_movieid(df, movie_id):
    # YOUR CODE HERE
    return return_value

def get_movieid_by_index(df, idx):
    # YOUR CODE HERE
    return return_value

The following tests your functions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Test with a random sample
for i in range(5):
    movie_id = movies3.sample()['id'].iloc[0]
    print(f"Movie id    : {movie_id}")
    print(f"Title       : {get_title_by_movieid(movies3, movie_id)}")
    idx = get_index_by_movieid(movies3,movie_id)
    print(f"Movie index : {idx}" )
    movie_id_recovered = get_movieid_by_index(movies3,idx)
    print(f"Check id    : {movie_id_recovered}")
    
    assert movie_id_recovered == movie_id
    print()

Now, given a movie id, we would like to recover its corresponding row in the `similarities` matrix. That row will be a list of similarities, one per index. Convert that to a dictionary in which keys are movie ids and values are similarities.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement "get_similarities_for_movieid".** It should obtain the corresponding row in a similarity matrix, and return a dictionary where keys are movie ids and values are similarities.</font>

In [ ]:
# Return all similarities to a given movie
def get_similarities_for_movieid(df, similarities, source_id):
    source_idx = get_index_by_movieid(df, source_id)
    sim_row = similarities[source_idx]
    output = {}
    
    # YOUR CODE HERE
    return output

The following code tests your function.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS
    
# Test your function
test_movie_id = 4256
print(f"Test movie_id {test_movie_id}, title '{movies3[movies3['id']==test_movie_id]['title'].values[0]}'")

# Call your function
sim_dictionary = get_similarities_for_movieid(movies3, cosine_sim, 4256)

# Samples some key, value pairs from the result
sample_size = 10
sim_dictionary_sample = dict(random.sample(list(sim_dictionary.items()), sample_size))

# Prints those
for other_movie_id, sim in sim_dictionary_sample.items():
    other_movie_title = movies3[movies3['id']==other_movie_id]['title'].values[0]
    print(f"{other_movie_id} ({other_movie_title}): {sim:.4f}")

## 3.3. Implement content-based recommendations <font color="#2680EB">[20%]</font>

Create a function `content_based_recommendations` that uses the similarity matrix  to find the top-k most recommended movies for a given movie. The result should be a dataframe with columns:

* `movie_id`: the id of the movie for which we are seeking recommendations.
* `recom_movie_id`: the id of the recommended movie.
* `title`: the title of the recommended movie.
* `keywords`: the keywords of the recommended movie.
* `similarity`: the similarity between movie_id and recom_movie_id

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement content based recommendations.**</font>

In [ ]:
def content_based_recommendations(movies, similarities, movie_id, num_movies=10):
    
    recommendations = []
    sim_dictionary = get_similarities_for_movieid(movies, similarities, movie_id)
    
    # YOUR CODE HERE
    
    return pd.DataFrame.from_records(recommendations,
                                     columns=['movie_id', 'recom_movie_id', 'similarity', 'title', 'keywords'])

The following tests your code.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Do not truncate any columns (the "keywords" column is quite large)
pd.set_option('display.max_colwidth', None)

# Obtain and display recommendations for a movie
display(content_based_recommendations(movies3, cosine_sim, 129, 10))

Now set `MOVIE_IDS` to be a list of three movies to be used for testing the content-based recommender.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# Set MOVIE_IDS to a list of three movie ids.
MOVIE_IDS = [...]

for movie_id in (MOVIE_IDS):
    display(content_based_recommendations(movies3, cosine_sim, movie_id, 11))


## 3.4. Questions about recommendations <font color="#2680EB">[20%]</font>

<font size="+1" color="#2680EB">**Answer the following questions.**</font>

**Regarding movie 1**

What is the title of the movie you tested?

* YOUR ANSWER HERE

Which movies you would say are NOT relevant in the list of recommended movie, if any?
* YOUR ANSWER HERE

What is the *accuracy* of the recommendations, i.e., what is the percentage of movies in the list that you would consider relevant?

* YOUR ANSWER HERE

**Regarding movie 2**
(Follow same scheme as above)

**Regarding movie 3**
(Follow same scheme as above)

<hr/>

**Regarding all the tested movies:**

Is there any of the movies for which you would say that an **infrequent keyword** (i.e., one having high inverse document frequency) substantially drives the results? Which movie and which keyword?

* YOUR ANSWER HERE

Considering the information available on this dataset, and assuming you do not have access to other information, how would you improve this content-based recommender system? One bullet point per idea, stating the idea and **why** do you think that would improve the recommendations.

* YOUR ANSWER HERE

# DELIVER (individually)

Remember to read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/data-mining-course/blob/master/upf/upf-evaluation.md).

Deliver a zip file containing:

* This notebook

## Extra points available

For more learning and extra points, use the **overview** field of the database, interpreted as 1- and 2-word ngrams. You may need to change the way in which you extract TF-IDF vectors to something like  `TfidfVectorizer(analyzer='word', ngram_range=(1,2), ...` and use the "overview" field instead of "keywords". Comment on whether the results are better or worse.

**Note:** if you go for the extra points, add ``<font size="+2" color="#2680EB">Additional results: overview</font>`` at the top of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the AI instructions stated above.</font>